# Datasets Lake — the 4 silver formats via polars

B1.8 seed notebook. Reads the **music** dataset's silver data in all four lakeFS formats — **Parquet / Arrow / Avro / Lance** — straight from the in-cluster lakeFS S3 gateway (no port-forward), proving every format is queryable from one tool. DuckDB SQL over the same files is the natural next step.

`LAKEFS_ENDPOINT` + `LAKEFS_ACCESS_KEY_ID` / `LAKEFS_SECRET_ACCESS_KEY` are injected into this pod by JupyterHub.

In [ ]:
import io, os
import polars as pl
import s3fs

LAKEFS_ENDPOINT = os.environ.get('LAKEFS_ENDPOINT', 'http://lakefs.data-mesh.svc.cluster.local:8000')
KEY = os.environ['LAKEFS_ACCESS_KEY_ID']
SECRET = os.environ['LAKEFS_SECRET_ACCESS_KEY']
REPO, BRANCH, TABLE = 'music', 'main', 'spotify_tracks'

# object_store options -> lakeFS S3 gateway (path-style, plain http)
SO = {'access_key_id': KEY, 'secret_access_key': SECRET, 'endpoint': LAKEFS_ENDPOINT,
      'allow_http': 'true', 'region': 'us-east-1'}
_fs = s3fs.S3FileSystem(key=KEY, secret=SECRET, use_ssl=False,
                        client_kwargs={'endpoint_url': LAKEFS_ENDPOINT})

def _first(fmt, ext):
    base = f'{REPO}/{BRANCH}/{fmt}/{TABLE}'
    return next(f for f in _fs.ls(base) if f.endswith(ext))

def read_parquet():
    return pl.read_parquet(f"s3://{_first('parquet', '.parquet')}", storage_options=SO)

def read_arrow():
    return pl.read_ipc(f"s3://{_first('arrow', '.arrow')}", storage_options=SO)

def read_avro():
    with _fs.open(_first('avro', '.avro'), 'rb') as f:
        return pl.read_avro(io.BytesIO(f.read()))

def read_lance():
    import lance
    ds = lance.dataset(f's3://{REPO}/{BRANCH}/lance/{TABLE}', storage_options=SO)
    return pl.from_arrow(ds.to_table())

print('lakeFS:', LAKEFS_ENDPOINT)

In [ ]:
for name, fn in [('parquet', read_parquet), ('arrow', read_arrow), ('avro', read_avro), ('lance', read_lance)]:
    try:
        df = fn()
        print(f'\n=== {name}: {df.height} rows x {df.width} cols ===')
        cols = [c for c in ('track_name', 'artists', 'danceability') if c in df.columns]
        if 'danceability' in df.columns and cols:
            display(df.select(cols).sort('danceability', descending=True).head(5))
        else:
            display(df.head(3))
    except Exception as e:
        print(f'\n=== {name}: FAILED -- {type(e).__name__}: {e} ===')

## DuckDB SQL over the same Parquet (bonus)
One engine, SQL over the lakeFS files.

In [ ]:
import duckdb
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"SET s3_endpoint='{LAKEFS_ENDPOINT.replace('http://','')}'; SET s3_use_ssl=false; SET s3_url_style='path'; SET s3_access_key_id='{KEY}'; SET s3_secret_access_key='{SECRET}';")
pq = _first('parquet', '.parquet')
con.execute(f"SELECT track_genre, count(*) n, avg(danceability) FROM read_parquet('s3://{pq}') GROUP BY 1 ORDER BY n DESC LIMIT 10").pl()